In [1]:
import time
from pathlib import Path
import numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
RAW = ROOT / "data" / "raw"; RAW.mkdir(parents=True, exist_ok=True)
GRID, HOURS, ROAD_THRESH = 8, 72, 0.7     # 0.7 = value picked by calibration earlier


def build_graph(rng, grid=GRID):
    G = nx.grid_2d_graph(grid, grid)
    nodes = list(G.nodes)
    rng.shuffle(nodes)
    shelters, depots = set(nodes[:8]), set(nodes[8:11])
    slope = rng.uniform(-1, 1, size=2)
    for (x, y) in G.nodes:
        elev = 5 + slope[0] * x / grid * 3 + slope[1] * y / grid * 3 + rng.normal(0, 0.4)
        G.nodes[(x, y)]["elevation"] = elev
        if (x, y) in shelters:
            G.nodes[(x, y)].update(ntype="shelter", population=0,
                                   capacity=int(rng.integers(2500, 7000)), inventory=0.0)
        elif (x, y) in depots:
            G.nodes[(x, y)].update(ntype="depot", population=0, capacity=0,
                                   inventory=float(rng.integers(8000, 20000)))
        else:
            pop = int(rng.integers(500, 4000))
            days = rng.uniform(1.5, 6.0)
            G.nodes[(x, y)].update(ntype="zone", population=pop, capacity=0,
                                   inventory=pop * 0.01 * 24 * days)
    for u, v in G.edges:
        G.edges[u, v]["length_km"] = float(rng.uniform(0.8, 2.5))
    return G


def make_rainfall(rng, hours=HOURS):
    t = np.arange(hours)
    base = rng.gamma(1.5, 0.6, size=hours)
    peak_t = rng.integers(12, 40)
    width = rng.uniform(4, 10)
    intensity = rng.uniform(15, 55)
    return base + intensity * np.exp(-0.5 * ((t - peak_t) / width) ** 2)


def add_failure_labels(df, horizon=6):
    # no groupby().apply(): new pandas drops the group columns
    df = df.sort_values(["scenario", "node", "hour"]).reset_index(drop=True)
    df["failed_now"] = ((df["isolated"] == 1) |
                        (df["days_of_supply"] < 0.25) |
                        (df["nearest_shelter_occ"] > 1.0)).astype(int)
    failed = df["failed_now"].to_numpy()
    tt = np.zeros(len(df))
    for idx in df.groupby(["scenario", "node"]).indices.values():
        f = failed[idx]
        T = len(f)
        out = np.full(T, float(T))          # censored: no failure observed
        nxt = None
        for i in range(T - 1, -1, -1):
            if f[i]:
                nxt = i
            if nxt is not None:
                out[i] = nxt - i
        tt[idx] = out
    df["hours_to_failure"] = tt
    df["will_fail_within_6h"] = (tt <= horizon).astype(int)
    return df


def run_simulation(seed, interventions=None, hours=HOURS, road_thresh=ROAD_THRESH):
    rng = np.random.default_rng(seed)
    G = build_graph(rng)
    rain = make_rainfall(rng, hours)
    interventions = interventions or {}

    for node, amt in interventions.get("extra_supply", {}).items():
        if node in G.nodes:
            G.nodes[node]["inventory"] += amt
    for node in interventions.get("open_shelter", []):
        if node in G.nodes:
            G.nodes[node].update(ntype="shelter", capacity=4000)
    forced_open = set(map(frozenset, interventions.get("restore_roads", [])))

    nodes = list(G.nodes)
    depth = {n: 0.0 for n in nodes}
    occupancy = {n: 0 for n in nodes}
    inventory = {n: G.nodes[n]["inventory"] for n in nodes}
    displaced_total = {n: 0 for n in nodes}
    rows = []

    for h in range(hours):
        new_depth = {}
        for n in nodes:
            nb = [depth[m] for m in G.neighbors(n)]
            inflow = rain[h] / 1000 * (1.0 + max(0, 5.2 - G.nodes[n]["elevation"]))
            spread = 0.15 * (np.mean(nb) - depth[n]) if nb else 0.0
            drain = 0.03 * depth[n] + 0.01
            new_depth[n] = max(0.0, depth[n] + inflow + spread - drain)
        depth = new_depth

        for u, v in G.edges:
            avg = (depth[u] + depth[v]) / 2
            G.edges[u, v]["failed"] = bool(avg > road_thresh and frozenset((u, v)) not in forced_open)

        open_G = nx.Graph([(u, v) for u, v in G.edges if not G.edges[u, v]["failed"]])
        open_G.add_nodes_from(nodes)
        shelters = [n for n in nodes if G.nodes[n]["ntype"] == "shelter"]

        comp = {}
        for i, c in enumerate(nx.connected_components(open_G)):
            for n in c:
                comp[n] = i
        shelter_comps = {comp[s] for s in shelters}

        hour_rows = []
        for n in nodes:
            nd = G.nodes[n]
            if nd["ntype"] != "zone":
                continue
            frac = min(1.0, max(0.0, (depth[n] - 0.3) / 1.5))
            displaced = int(nd["population"] * frac)
            newly = max(0, displaced - displaced_total[n])
            displaced_total[n] = max(displaced_total[n], displaced)
            requests = int(rng.poisson(0.002 * nd["population"] * depth[n] * 3))

            consumption = (nd["population"] + 0.5 * displaced) * 0.01
            inventory[n] = max(0.0, inventory[n] - consumption)
            base_daily = max(nd["population"] * 0.01 * 24, 1e-9)

            nearest, dist = None, np.inf
            if comp[n] in shelter_comps:
                cands = [s for s in shelters if comp[s] == comp[n]]
                nearest = min(cands, key=lambda s: nx.shortest_path_length(open_G, n, s))
                dist = nx.shortest_path_length(open_G, n, nearest)
                if newly > 0:
                    occupancy[nearest] += newly

            open_roads = sum(not G.edges[n, m]["failed"] for m in G.neighbors(n))
            hour_rows.append(dict(
                scenario=seed, hour=h, node=f"{n[0]}_{n[1]}", x=n[0], y=n[1],
                population=nd["population"], elevation=nd["elevation"],
                rain_mm=rain[h], flood_depth=depth[n],
                displaced=displaced, requests=requests,
                inventory=inventory[n],
                days_of_supply=inventory[n] / base_daily,
                open_road_frac=open_roads / max(1, G.degree(n)),
                isolated=int(nearest is None),
                shelter_dist=dist if np.isfinite(dist) else 99,
                nearest_shelter_occ=0.0,
                _shelter=nearest,
            ))

        # each zone gets the occupancy of its OWN nearest shelter (not the global max)
        ratio = {s: occupancy[s] / max(1, G.nodes[s]["capacity"]) for s in shelters}
        for r in hour_rows:
            s = r.pop("_shelter")
            r["nearest_shelter_occ"] = ratio[s] if s is not None else 0.0
        rows.extend(hour_rows)

    return add_failure_labels(pd.DataFrame(rows))


print("simulator ready:", callable(run_simulation), "| data folder:", RAW)

simulator ready: True | data folder: /home/ibm-user/floodguard/data/raw


In [3]:
import math
from IPython.display import display

# ---- 1) exact replay of flood depth at every node and closed roads at every hour ----
def replay_state(seed, road_thresh=ROAD_THRESH):
    rng = np.random.default_rng(seed)
    G = build_graph(rng); rain = make_rainfall(rng)       # same draws as run_simulation
    nodes = list(G.nodes); depth = {n: 0.0 for n in nodes}; trace = []
    for h in range(HOURS):
        new = {}
        for n in nodes:
            nb = [depth[m] for m in G.neighbors(n)]
            inflow = rain[h] / 1000 * (1.0 + max(0, 5.2 - G.nodes[n]["elevation"]))
            spread = 0.15 * (np.mean(nb) - depth[n]) if nb else 0.0
            new[n] = max(0.0, depth[n] + inflow + spread - 0.03 * depth[n] - 0.01)
        depth = new
        closed = {frozenset((u, v)) for u, v in G.edges if (depth[u] + depth[v]) / 2 > road_thresh}
        trace.append({"depth": dict(depth), "closed": closed})
    return G, trace

def isolated_zones(G, closed):
    H = nx.Graph(); H.add_nodes_from(G.nodes)
    H.add_edges_from((u, v) for u, v in G.edges if frozenset((u, v)) not in closed)
    reach = set()
    for s in (n for n in G.nodes if G.nodes[n]["ntype"] == "shelter"):
        reach |= nx.node_connected_component(H, s)
    return {n for n in G.nodes if G.nodes[n]["ntype"] == "zone" and n not in reach}

# ---- 2) check the replay against your saved data ----
pq, csv = RAW / "sim_runs.parquet", RAW / "sim_runs.csv"
sim = pd.read_parquet(pq) if pq.exists() else pd.read_csv(csv)

def verify(seed):
    G, tr = replay_state(seed)
    g = sim[sim["scenario"] == seed]; err, bad = 0.0, 0
    for h, grp in g.groupby("hour"):
        iso = isolated_zones(G, tr[int(h)]["closed"])
        for r in grp.itertuples():
            n = (int(r.x), int(r.y))
            err = max(err, abs(float(r.flood_depth) - tr[int(h)]["depth"][n]))
            bad += int((n in iso) != bool(r.isolated))
    return err, bad

for s in (0, 7, 150):
    e, b = verify(s)
    print(f"seed {s}: max depth error {e:.2e} | isolation mismatches {b}", "OK" if e < 1e-4 and b == 0 else "CHECK")

# ---- 3) hospitals (assumption) and two kinds of route ----
def hospital_nodes(G, k=3):
    """ASSUMPTION: hospitals sit in the k most populated zones. Replace with real data when available."""
    zones = [n for n in G.nodes if G.nodes[n]["ntype"] == "zone"]
    return sorted(zones, key=lambda n: -G.nodes[n]["population"])[:k]

def travel_graph(G, st):
    H = nx.Graph()
    for u, v in G.edges:
        if frozenset((u, v)) in st["closed"]: continue
        d = (st["depth"][u] + st["depth"][v]) / 2
        speed = 40 * (1 - min(0.8, d))                              # km/h, ASSUMPTION: slower in deeper water
        H.add_edge(u, v, minutes=60 * G.edges[u, v]["length_km"] / speed,
                   risk=G.edges[u, v]["length_km"] * (d + 0.05) + (5 if d > 0.3 else 0), depth=d)
    return H

def routes(G, st, origin, hospitals):
    """Fastest = least minutes. Safest = least flood exposure. Origin is the patient's zone."""
    H, out = travel_graph(G, st), {}
    for name, wt in (("fastest", "minutes"), ("safest", "risk")):
        best = None
        for hp in hospitals:
            if origin in H and hp in H:
                try: c, p = nx.single_source_dijkstra(H, origin, hp, weight=wt)
                except nx.NetworkXNoPath: continue
                if best is None or c < best[0]: best = (c, p, hp)
        if best:
            p = best[1]; legs = list(zip(p, p[1:]))
            out[name] = dict(path=p, hospital=best[2], minutes=sum(H[a][b]["minutes"] for a, b in legs),
                             max_depth=max((H[a][b]["depth"] for a, b in legs), default=0.0))
        else:
            out[name] = None                                         # no open road reaches any hospital
    return out

G, tr = replay_state(150); hs = hospital_nodes(G)
iso = sorted(isolated_zones(G, tr[30]["closed"]))
print("hospitals:", hs, "| zones cut off at hour 30:", len(iso))
origin = next((n for n in G.nodes if G.nodes[n]["ntype"] == "zone" and n not in iso), None)
r = routes(G, tr[30], origin, hs)
for k, v in r.items():
    print(k, "->", "NO ROUTE" if v is None else f"{v['minutes']:.0f} min, max depth {v['max_depth']:.2f} m, {len(v['path'])-1} segments")

# ---- 4) optional map placement (illustrative only) ----
def to_latlon(node, lat0, lon0, km=1.65):
    """Puts grid cell (x, y) on a map. lat0/lon0 = south-west corner of the area you choose."""
    return lat0 + node[1] * km / 111.0, lon0 + node[0] * km / (111.0 * math.cos(math.radians(lat0)))

seed 0: max depth error 3.33e-16 | isolation mismatches 0 OK
seed 7: max depth error 2.50e-16 | isolation mismatches 0 OK
seed 150: max depth error 2.78e-16 | isolation mismatches 0 OK
hospitals: [(2, 0), (0, 7), (6, 2)] | zones cut off at hour 30: 0
fastest -> 8 min, max depth 0.37 m, 2 segments
safest -> 19 min, max depth 0.28 m, 7 segments
